<a href="https://colab.research.google.com/github/iiiyahiya111/pi_explore/blob/main/test_ml_pi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install scikit-learn

import json
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# 1. Load a small, built-in dataset
iris = load_iris()
X = iris.data
y = iris.target

# 2. Split training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Normalize the inputs
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. Train a small neural network: 4 -> 8 -> 3
mlp = MLPClassifier(
    hidden_layer_sizes=(8,),
    activation="relu",
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)

mlp.fit(X_train, y_train)

# 5. Evaluate the model
accuracy = accuracy_score(y_test, mlp.predict(X_test))
print(f"Test accuracy: {accuracy:.4f}")

# 6. Export the model parameters to JSON
model_data = {
    "input_mean": scaler.mean_.tolist(),
    "input_scale": scaler.scale_.tolist(),
    "weights": [w.tolist() for w in mlp.coefs_],
    "biases": [b.tolist() for b in mlp.intercepts_],
    "classes": mlp.classes_.tolist(),
    "class_names": iris.target_names.tolist()
}

with open("iris_mlp_model.json", "w") as f:
    json.dump(model_data, f)

print("Model exported: iris_mlp_model.json")

# 7. Download the model to your Windows laptop
from google.colab import files
files.download("iris_mlp_model.json")

Test accuracy: 0.9667
Model exported: iris_mlp_model.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **file in ras pi**

In [ ]:
import json
import math
from pathlib import Path

model_path = Path(__file__).with_name("iris_mlp_model.json")

with open(model_path, "r") as f:
    model = json.load(f)


def dense_layer(values, weights, biases):
    outputs = []

    for j, bias in enumerate(biases):
        total = bias

        for i, value in enumerate(values):
            total += value * weights[i][j]

        outputs.append(total)

    return outputs


def predict(features):
    # Use the same normalization learned during training.
    values = [
        (value - mean) / scale
        for value, mean, scale in zip(
            features,
            model["input_mean"],
            model["input_scale"]
        )
    ]

    # Forward pass through the trained neural network.
    for layer, (weights, biases) in enumerate(
        zip(model["weights"], model["biases"])
    ):
        values = dense_layer(values, weights, biases)

        # ReLU on the hidden layer only.
        if layer < len(model["weights"]) - 1:
            values = [max(0.0, value) for value in values]

    # Stable softmax for the three output classes.
    max_value = max(values)
    exponentials = [math.exp(v - max_value) for v in values]
    total = sum(exponentials)
    probabilities = [v / total for v in exponentials]

    best = max(
        range(len(probabilities)),
        key=lambda i: probabilities[i]
    )

    class_id = int(model["classes"][best])
    class_name = model["class_names"][class_id]

    return class_name, probabilities[best]


if __name__ == "__main__":
    # Example flower measurements in centimetres.
    sample = [5.1, 3.5, 1.4, 0.2]

    label, confidence = predict(sample)

    print("Input:", sample)
    print("Predicted class:", label)
    print(f"Model score: {confidence:.4f}")
